# One-Electron Integrals and Basis Sets

This notebook demonstrates LibAccInt's one-electron integral APIs and basis set handling.

**What you'll learn:**
- Defining molecules with `Atom` and loading basis sets
- Computing overlap ($S$), kinetic ($T$), and nuclear attraction ($V$) matrices
- Building the core Hamiltonian $H = T + V$
- Comparing basis sets (size, condition number)
- Composing operators with `OneElectronOperator`

In [ ]:
import numpy as np
import libaccint as lai

print(f"LibAccInt version: {lai.__version__}")
print(f"OpenMP available:  {lai.has_openmp()}")
print(f"CUDA backend:      {lai.has_cuda_backend()}")
print(f"HIP backend:       {lai.has_hip_backend()}")

# The Engine automatically routes work to the best available backend.
# All compute methods accept a BackendHint to override the default:
#   Auto (default) — engine decides    PreferCPU / PreferGPU — soft preference
#   ForceCPU / ForceGPU — hard constraint
print(f"\nBackendHint options: {[h.name for h in lai.BackendHint]}")

## 1. Defining a Molecule

Atoms are specified by atomic number and Cartesian position **in Bohr** (libaccint's native unit).
We use the standard water geometry: $r_{\text{OH}} = 1.809$ Bohr, $\angle_{\text{HOH}} = 104.5°$.

In [ ]:
atoms = [
    lai.Atom(8, [0.0, 0.0, 0.0]),                  # O at origin
    lai.Atom(1, [0.0,  1.43233673, -1.10866041]),   # H
    lai.Atom(1, [0.0, -1.43233673, -1.10866041]),   # H
]

n_elec = sum(a.atomic_number for a in atoms)
print(f"H\u2082O: {len(atoms)} atoms, {n_elec} electrons")
for a in atoms:
    p = a.position
    print(f"  Z={a.atomic_number:2d}  ({p.x:10.6f}, {p.y:10.6f}, {p.z:10.6f})")

## 2. Exploring Basis Sets

LibAccInt ships 40+ basis sets from the Basis Set Exchange. Use `list_available_basis_sets()` to see them all.

In [ ]:
available = lai.list_available_basis_sets()
print(f"{len(available)} basis sets available:")
for name in available:
    print(f"  {name}")

In [ ]:
basis = lai.basis_set("STO-3G", atoms)

print(f"STO-3G for H\u2082O:")
print(f"  Shells:          {basis.n_shells()}")
print(f"  Basis functions: {basis.n_basis_functions()}")
print(f"  Max AM:          {basis.max_angular_momentum()}")

In [ ]:
am_labels = ['s', 'p', 'd', 'f', 'g']

for i in range(basis.n_shells()):
    s = basis.shell(i)
    c = s.center()
    print(f"Shell {i}: {am_labels[s.angular_momentum()]}-type, "
          f"{s.n_primitives()} prims, {s.n_functions()} funcs, "
          f"center=({c.x:.4f}, {c.y:.4f}, {c.z:.4f})")
    print(f"  exponents:    {s.exponents()}")
    print(f"  coefficients: {s.coefficients()}")

## 3. Overlap Matrix

The overlap integral $S_{\mu\nu} = \langle \mu | \nu \rangle$ measures how much two basis functions overlap in space.
Diagonal elements are always 1 (normalized functions), and the matrix is symmetric.

In [ ]:
# Engine(basis) creates a hybrid engine: CpuEngine always, plus CudaEngine if available.
# BackendHint.Auto (the default) lets the engine decide where to run each operation.
engine = lai.Engine(basis)
print(f"GPU available in engine: {engine.gpu_available()}")

S = engine.compute_overlap_matrix()  # hint defaults to Auto
nbf = basis.n_basis_functions()

print(f"\nOverlap matrix S: shape {S.shape}")
print(f"Symmetric: {np.allclose(S, S.T)}")
print(f"Diagonal = 1: {np.allclose(np.diag(S), 1.0)}")
print(f"\nS =\n{np.array2string(S, precision=4, suppress_small=True)}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(S, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_title("Overlap Matrix $S$ (H\u2082O / STO-3G)")
ax.set_xlabel("Basis function")
ax.set_ylabel("Basis function")
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

## 4. Kinetic and Nuclear Attraction Matrices

$$T_{\mu\nu} = \langle \mu | -\tfrac{1}{2}\nabla^2 | \nu \rangle, \qquad V_{\mu\nu} = \langle \mu | -\sum_A \frac{Z_A}{|\mathbf{r} - \mathbf{R}_A|} | \nu \rangle$$

In [ ]:
T = engine.compute_kinetic_matrix()
V = engine.compute_nuclear_matrix(atoms)

print(f"Kinetic T: shape {T.shape}, symmetric: {np.allclose(T, T.T)}")
print(f"Nuclear V: shape {V.shape}, symmetric: {np.allclose(V, V.T)}")
print(f"\nT =\n{np.array2string(T, precision=4, suppress_small=True)}")
print(f"\nV =\n{np.array2string(V, precision=4, suppress_small=True)}")

## 5. Core Hamiltonian

The core (one-electron) Hamiltonian is $H = T + V$. LibAccInt can compute it directly or you can sum the pieces.

In [ ]:
H = engine.compute_core_hamiltonian(atoms)
H_manual = T + V

print(f"H from engine matches T + V: {np.allclose(H, H_manual)}")
print(f"Max |H - (T+V)|: {np.max(np.abs(H - H_manual)):.2e}")
print(f"\nH =\n{np.array2string(H, precision=4, suppress_small=True)}")

## 6. Basis Set Comparison

Larger basis sets have more functions, better accuracy, but higher computational cost.
The condition number of $S$ indicates linear dependence — higher values mean near-redundant functions.

In [ ]:
basis_names = ["STO-3G", "6-31G*", "cc-pVDZ"]

print(f"{'Basis':<12} {'nbf':>5} {'shells':>7} {'max AM':>7} {'cond(S)':>12}")
print("-" * 48)
for name in basis_names:
    b = lai.basis_set(name, atoms)
    e = lai.Engine(b)
    S_b = e.compute_overlap_matrix()
    cond = np.linalg.cond(S_b)
    print(f"{name:<12} {b.n_basis_functions():>5} {b.n_shells():>7} "
          f"{b.max_angular_momentum():>7} {cond:>12.1f}")

## 7. Composite Operators with `OneElectronOperator`

You can compose multiple one-electron operators into a single computation.
This computes $H = T + V$ in one engine pass instead of two.

In [ ]:
# Build nuclear attraction operator parameters
nuc_params = lai.PointChargeParams(
    [a.position.x for a in atoms],
    [a.position.y for a in atoms],
    [a.position.z for a in atoms],
    [float(a.atomic_number) for a in atoms]
)

# Compose T + V into one operator
op_hcore = lai.OneElectronOperator(lai.Operator.kinetic())
op_hcore.add(lai.Operator.nuclear(nuc_params))
print(f"Composite operator: {op_hcore.n_contributions()} contributions")

# Compute in one pass
H_composite = engine.compute_1e_parallel(op_hcore)
print(f"Matches compute_core_hamiltonian: {np.allclose(H_composite, H)}")
print(f"Max difference: {np.max(np.abs(H_composite - H)):.2e}")

## 8. Backend Hints: CPU, GPU, and Hybrid Execution

Every compute method accepts a `BackendHint` parameter. The `Engine` contains both
a CPU and (optionally) a GPU backend — `BackendHint` controls routing:

| Hint | Behaviour |
|------|-----------|
| `Auto` (default) | Engine picks the best backend per operation |
| `PreferCPU` / `PreferGPU` | Soft preference; falls back if unavailable |
| `ForceCPU` / `ForceGPU` | Hard constraint; raises if unavailable |

The same notebook code works unchanged on CPU-only, GPU-only, and hybrid builds.

In [ ]:
# Explicitly request CPU — always works
S_cpu = engine.compute_overlap_matrix(lai.BackendHint.ForceCPU)

# Request GPU with automatic fallback to CPU if no GPU is present
S_auto = engine.compute_overlap_matrix(lai.BackendHint.PreferGPU)

print(f"ForceCPU matches Auto: {np.allclose(S_cpu, S)}")
print(f"PreferGPU matches Auto: {np.allclose(S_auto, S)}")

if engine.gpu_available():
    S_gpu = engine.compute_overlap_matrix(lai.BackendHint.ForceGPU)
    print(f"ForceGPU matches Auto: {np.allclose(S_gpu, S)}")
else:
    print("(No GPU — ForceGPU would raise BackendError, PreferGPU falls back to CPU)")

## Summary

| API | Purpose |
|-----|--------|
| `Atom(Z, pos)` | Define an atom |
| `basis_set(name, atoms)` | Load a basis set |
| `list_available_basis_sets()` | Show available basis sets |
| `Engine(basis)` | Create the hybrid CPU+GPU integral engine |
| `compute_overlap_matrix(hint)` | Overlap $S$ |
| `compute_kinetic_matrix(hint)` | Kinetic $T$ |
| `compute_nuclear_matrix(atoms, hint)` | Nuclear attraction $V$ |
| `compute_core_hamiltonian(atoms, hint)` | Core Hamiltonian $H = T + V$ |
| `OneElectronOperator` | Compose operators for single-pass computation |
| `BackendHint.Auto / PreferGPU / ForceCPU` | Control CPU vs GPU routing |

**Next:** [02_work_units_and_integral_buffers.ipynb](02_work_units_and_integral_buffers.ipynb) — low-level work-unit APIs.